# Normalización y EDA de impagos

Este notebook trabaja con archivos `CSV` y:

- Carga `Variable Y.csv`
- Convierte `VERDADERO` a `1` y `FALSO` a `0`
- Trata valores vacíos y `#¡NULO!` como ausentes
- Calcula el porcentaje de impagos total, por mes/año y por año

In [ ]:
# Ejecuta esta celda si tu entorno no tiene las librerías instaladas.
%pip install pandas numpy matplotlib seaborn

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)

ruta_datos = Path('Variable Y.csv')

df = pd.read_csv(ruta_datos, sep=';', encoding='utf-8-sig')

# El archivo trae columnas sin nombre o vacías extra; se eliminan.
df = df.loc[:, ~df.columns.astype(str).str.match(r'^Unnamed')]
df = df.loc[:, df.columns.astype(str).str.strip() != '']

# Además, se eliminan columnas completamente vacías por seguridad.
df = df.dropna(axis=1, how='all')

# Si hubiera filas completamente vacías, también se eliminan.
df = df.dropna(axis=0, how='all').copy()

print(f'Filas: {df.shape[0]}, columnas: {df.shape[1]}')
df.head()

In [ ]:
df_normalizado = df.copy()

columnas_periodo = [col for col in df_normalizado.columns if col != 'ID']

mapeo_binario = {
    'VERDADERO': 1,
    'FALSO': 0,
    '#¡NULO!': np.nan,
    '': np.nan
}

df_normalizado[columnas_periodo] = df_normalizado[columnas_periodo].replace(mapeo_binario)
df_normalizado['ID'] = pd.to_numeric(df_normalizado['ID'], errors='coerce')

# Convierte las columnas mensuales a tipo numérico para asegurar 0/1.
df_normalizado[columnas_periodo] = df_normalizado[columnas_periodo].apply(pd.to_numeric, errors='coerce')

df_normalizado.head()

In [ ]:
# Exportación opcional del resultado normalizado.
ruta_salida = Path('Variable_Y_normalizada.csv')
df_normalizado.to_csv(ruta_salida, index=False)
print(f'Archivo guardado en: {ruta_salida.resolve()}')

In [ ]:
meses_es = {
    'Enero': '01',
    'Febrero': '02',
    'Marzo': '03',
    'Abril': '04',
    'Mayo': '05',
    'Junio': '06',
    'Julio': '07',
    'Agosto': '08',
    'Septiembre': '09',
    'Octubre': '10',
    'Noviembre': '11',
    'Diciembre': '12'
}

df_largo = df_normalizado.melt(
    id_vars='ID',
    value_vars=columnas_periodo,
    var_name='periodo',
    value_name='impago'
)

periodo_split = df_largo['periodo'].str.extract(r'(?P<mes>\w+)\s+(?P<anio>\d{4})')
df_largo['mes'] = periodo_split['mes']
df_largo['anio'] = pd.to_numeric(periodo_split['anio'], errors='coerce')
df_largo['fecha'] = pd.to_datetime(
    periodo_split['anio'].astype(str) + '-' + periodo_split['mes'].map(meses_es) + '-01',
    errors='coerce'
)

porcentaje_total = df_largo['impago'].mean() * 100

porcentaje_por_mes_anio = (
    df_largo.dropna(subset=['fecha'])
    .groupby('fecha', as_index=False)['impago']
    .mean()
    .assign(porcentaje_impago=lambda x: x['impago'] * 100)
    .sort_values('fecha')
)

porcentaje_por_anio = (
    df_largo.dropna(subset=['anio'])
    .groupby('anio', as_index=False)['impago']
    .mean()
    .assign(porcentaje_impago=lambda x: x['impago'] * 100)
    .sort_values('anio')
)

print(f'Porcentaje total de impagos: {porcentaje_total:.2f}%')

display(
    porcentaje_por_mes_anio[['fecha', 'porcentaje_impago']]
    .rename(columns={'fecha': 'mes_anio', 'porcentaje_impago': 'pct_impago'})
)

display(
    porcentaje_por_anio[['anio', 'porcentaje_impago']]
    .rename(columns={'porcentaje_impago': 'pct_impago'})
)

fig, axes = plt.subplots(1, 2, figsize=(18, 5))

sns.barplot(
    data=porcentaje_por_mes_anio,
    x='fecha',
    y='porcentaje_impago',
    color='#D55E00',
    ax=axes[0]
)
axes[0].set_title('Porcentaje de impagos por mes/año')
axes[0].set_xlabel('Mes/Año')
axes[0].set_ylabel('% de impago')
axes[0].tick_params(axis='x', rotation=45)

sns.barplot(
    data=porcentaje_por_anio,
    x='anio',
    y='porcentaje_impago',
    color='#0072B2',
    ax=axes[1]
)
axes[1].set_title('Porcentaje de impagos por año')
axes[1].set_xlabel('Año')
axes[1].set_ylabel('% de impago')

plt.tight_layout()
plt.show()

## EDA de la tabla EEFF

Esta sección carga la tabla `EEFF` desde `CSV`, limpia marcadores de ausentes como `n.d.` y resume los tipos de variables para distinguir identificadores, variables categóricas y variables numéricas.


In [ ]:
ruta_eeff = Path('EEFF base de datos.csv')

df_eeff = pd.read_csv(ruta_eeff, sep=';', encoding='utf-8-sig')
df_eeff = df_eeff.loc[:, ~df_eeff.columns.astype(str).str.match(r'^Unnamed')]
df_eeff = df_eeff.loc[:, df_eeff.columns.astype(str).str.strip() != '']
with pd.option_context('future.no_silent_downcasting', True):
    df_eeff = df_eeff.replace({'n.d.': np.nan, 'N.D.': np.nan, '': np.nan})
df_eeff = df_eeff.infer_objects(copy=False)

def parsear_numero_eeff(valor):
    if pd.isna(valor):
        return np.nan

    texto = str(valor).strip()
    if texto == '':
        return np.nan

    if '.' in texto and ',' in texto:
        texto = texto.replace('.', '').replace(',', '.')
    elif ',' in texto:
        texto = texto.replace(',', '.')
    elif '.' in texto:
        if pd.Series([texto]).str.match(r'^-?\d{1,3}(\.\d{3})+$').iloc[0]:
            texto = texto.replace('.', '')

    try:
        return float(texto)
    except ValueError:
        return np.nan

print(f'EEFF -> filas: {df_eeff.shape[0]}, columnas: {df_eeff.shape[1]}')
df_eeff.head()


In [ ]:
if 'eeff_con_variable_y' not in globals():
    ruta_eeff_con_variable_y = Path('EEFF_con_variable_Y.csv')
    eeff_con_variable_y = pd.read_csv(ruta_eeff_con_variable_y, encoding='utf-8-sig')

ids_con_4_registros = eeff_con_variable_y['ID'].value_counts()
ids_con_4_registros = ids_con_4_registros[ids_con_4_registros == 4].index

df_4 = eeff_con_variable_y[eeff_con_variable_y['ID'].isin(ids_con_4_registros)].copy()

print(f'df_4 -> filas: {df_4.shape[0]}, IDs únicos: {df_4["ID"].nunique()}')
df_4.head()


In [ ]:
df_eeff_tipado = df_eeff.copy()

for col in df_eeff_tipado.columns:
    if df_eeff_tipado[col].dtype == 'object':
        serie_original = df_eeff_tipado[col]
        serie_parseada = serie_original.map(parsear_numero_eeff)
        no_nulos_original = serie_original.notna().sum()
        ratio_conversion = 0 if no_nulos_original == 0 else serie_parseada.notna().sum() / no_nulos_original

        if ratio_conversion >= 0.95:
            df_eeff_tipado[col] = serie_parseada

def clasificar_variable(serie, nombre_columna):
    serie_no_nula = serie.dropna()
    nombre = str(nombre_columna).strip().lower()

    if nombre in {'id', 'identifier'} or nombre.startswith('id'):
        return 'Identificador'

    if pd.api.types.is_numeric_dtype(serie):
        if serie_no_nula.empty:
            return 'Numérica'

        valores_unicos = serie_no_nula.nunique()
        es_entera = np.all(np.isclose(serie_no_nula % 1, 0))

        if valores_unicos == 2:
            return 'Binaria numérica'
        if es_entera and valores_unicos <= 20:
            return 'Numérica discreta'
        if es_entera:
            return 'Numérica entera'
        return 'Numérica continua'

    valores_unicos = serie_no_nula.nunique()
    if valores_unicos == 2:
        return 'Binaria categórica'
    return 'Categórica'

resumen_variables = pd.DataFrame({
    'variable': df_eeff_tipado.columns,
    'dtype_pandas': [str(df_eeff_tipado[col].dtype) for col in df_eeff_tipado.columns],
    'tipo_variable': [clasificar_variable(df_eeff_tipado[col], col) for col in df_eeff_tipado.columns],
    'nulos_pct': [df_eeff_tipado[col].isna().mean() * 100 for col in df_eeff_tipado.columns],
    'valores_unicos': [df_eeff_tipado[col].nunique(dropna=True) for col in df_eeff_tipado.columns],
    'ejemplo_valores': [', '.join(map(str, df_eeff_tipado[col].dropna().astype(str).unique()[:3])) for col in df_eeff_tipado.columns]
}).sort_values(['tipo_variable', 'nulos_pct', 'valores_unicos'], ascending=[True, False, False])

display(resumen_variables)

display(
    resumen_variables.groupby('tipo_variable', as_index=False)
    .agg(numero_variables=('variable', 'count'))
    .sort_values('numero_variables', ascending=False)
)


In [ ]:
conteo_tipos = (
    resumen_variables['tipo_variable']
    .value_counts()
    .rename_axis('tipo_variable')
    .reset_index(name='numero_variables')
)

plt.figure(figsize=(10, 5))
sns.barplot(data=conteo_tipos, x='tipo_variable', y='numero_variables', palette='Blues_r')
plt.title('Distribuci?n de tipos de variables en EEFF')
plt.xlabel('Tipo de variable')
plt.ylabel('N?mero de variables')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

resumen_variables.groupby('tipo_variable')['nulos_pct'].describe().round(2)
